In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [ ]:
cdf = spark.read.csv("cust3.csv", header=True, inferSchema=True)
pdf = spark.read.csv("prod3.csv", header=True, inferSchema=True)
odf = spark.read.csv("ord3.csv", header=True, inferSchema=True)

In [ ]:
cdf.show()
pdf.show()
odf.show()

+-----------+------+---------+---+-----------------+
|customer_id|  name|     city|age|registration_date|
+-----------+------+---------+---+-----------------+
|          1|  Amit|    Delhi| 25|       2023-01-10|
|          2|  Sara|   Mumbai| 30|       2022-12-05|
|          3|  John|Bangalore| 28|       2023-02-15|
|          4|   Ali|Hyderabad| 35|       2021-11-20|
|          5|  Neha|  Chennai| 27|       2023-03-01|
|          6|  Ravi|    Delhi| 40|       2020-07-19|
|          7| Priya|   Mumbai| 22|       2023-04-12|
|          8| Kiran|Hyderabad| 31|       2022-05-22|
|          9| David|Bangalore| 29|       2023-01-25|
|         10| Meena|  Chennai| 33|       2021-09-10|
|         11| Arjun|    Delhi| 26|       2022-08-18|
|         12|Fatima|Hyderabad| 24|       2023-02-28|
|         13| Rahul|   Mumbai| 37|       2020-06-30|
|         14| Sneha|  Chennai| 21|       2023-03-14|
|         15|Vikram|Bangalore| 45|       2019-12-11|
|         16|  Zoya|    Delhi| 23|       2023-

Top 3 Customers by Total Spending

In [ ]:
combined_df=cdf.join(odf,"customer_id").join(pdf,"product_id")
grouped_df=combined_df.groupBy("customer_id","name").agg(sum(odf.quantity*pdf.price).alias("spending"))
window_spec = Window.orderBy(col("spending").desc())
result = grouped_df.withColumn("rank",dense_rank().over(window_spec)).filter(col("rank")<=3)
result.show()



+-----------+----+--------+----+
|customer_id|name|spending|rank|
+-----------+----+--------+----+
|          4| Ali|  140000|   1|
|          1|Amit|   74000|   2|
|          2|Sara|   63000|   3|
+-----------+----+--------+----+



Running Total of Customer Spending by date

In [ ]:
joined_df = cdf.join(odf,"customer_id").join(pdf,"product_id")
grouped_df = joined_df.groupBy("customer_id","name","order_date").agg(sum(odf.quantity*pdf.price).alias("spending"))
window_spec = Window.partitionBy("customer_id").orderBy("order_date")
result = grouped_df.withColumn("running_total",sum("spending").over(window_spec))
result.show()

+-----------+-----+----------+--------+-------------+
|customer_id| name|order_date|spending|running_total|
+-----------+-----+----------+--------+-------------+
|          1| Amit|2023-05-01|   70000|        70000|
|          1| Amit|2023-05-05|    4000|        74000|
|          2| Sara|2023-05-03|   60000|        60000|
|          2| Sara|2023-05-11|    3000|        63000|
|          3| John|2023-05-04|    6000|         6000|
|          4|  Ali|2023-05-06|  140000|       140000|
|          5| Neha|2023-05-07|    1000|         1000|
|          6| Ravi|2023-05-08|   25000|        25000|
|          7|Priya|2023-05-09|    2400|         2400|
|          8|Kiran|2023-05-10|    2500|         2500|
+-----------+-----+----------+--------+-------------+



Most Sold Product in Each Category

In [ ]:
joined_df=pdf.join(odf,"product_id")
grouped_df=joined_df.groupBy("category","product_id","product_name").agg(sum(odf.quantity).alias("total_sold"))
window_spec=Window.partitionBy("category").orderBy(col("total_sold").desc())
result=grouped_df.withColumn("rank",dense_rank().over(window_spec)).filter(col("rank")==1)\
.select("category","product_name","product_id","total_sold")
result.show()

+-----------+------------+----------+----------+
|   category|product_name|product_id|total_sold|
+-----------+------------+----------+----------+
|Accessories|     Charger|       112|         5|
|Electronics|      Laptop|       101|         3|
|    Fashion|       Jeans|       111|         1|
|    Fashion|       Shoes|       105|         1|
|    Fashion|     T-shirt|       110|         1|
+-----------+------------+----------+----------+



monthly revnue trend

In [ ]:
joined_df = pdf.join(odf,"product_id")
grouped_df = joined_df.groupBy(month("order_date").alias("month")).agg(sum(col("quantity")*col("price")).alias("revenue"))
window_spec = Window.orderBy(col("month"))
result = grouped_df.withColumn("previous_month_revenue",lag("revenue").over(window_spec))
result.show()

+-----+-------+----------------------+
|month|revenue|previous_month_revenue|
+-----+-------+----------------------+
|    5| 313900|                  NULL|
+-----+-------+----------------------+



percentage contributuion of each product

In [ ]:
joined_df = pdf.join(odf,"product_id")

grouped_df = joined_df.groupBy("product_id","product_name").agg(sum(col("quantity")*col("price"))\
.alias("each_product_revenue"))

total_revenue = grouped_df.agg(sum("each_product_revenue")).collect()[0][0]

result = grouped_df.withColumn("percentage_contribution",round((col("each_product_revenue")/total_revenue)*100,2))

result.show()

+----------+------------+--------------------+-----------------------+
|product_id|product_name|each_product_revenue|percentage_contribution|
+----------+------------+--------------------+-----------------------+
|       103|      Tablet|               25000|                   7.96|
|       111|       Jeans|                2500|                    0.8|
|       105|       Shoes|                4000|                   1.27|
|       102|      Mobile|               60000|                  19.11|
|       101|      Laptop|              210000|                   66.9|
|       108|    Keyboard|                2400|                   0.76|
|       104|  Headphones|                6000|                   1.91|
|       110|     T-shirt|                1000|                   0.32|
|       112|     Charger|                3000|                   0.96|
+----------+------------+--------------------+-----------------------+



In [ ]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy()

result = grouped_df.withColumn(
    "total_revenue",
    sum("each_product_revenue").over(window_spec)
).withColumn(
    "percentage_contribution",
    round(
        col("each_product_revenue") * 100 / col("total_revenue"),
        2
    )
).drop("total_revenue")

result.show()

+----------+------------+--------------------+-----------------------+
|product_id|product_name|each_product_revenue|percentage_contribution|
+----------+------------+--------------------+-----------------------+
|       103|      Tablet|               25000|                   7.96|
|       111|       Jeans|                2500|                    0.8|
|       105|       Shoes|                4000|                   1.27|
|       102|      Mobile|               60000|                  19.11|
|       101|      Laptop|              210000|                   66.9|
|       108|    Keyboard|                2400|                   0.76|
|       104|  Headphones|                6000|                   1.91|
|       110|     T-shirt|                1000|                   0.32|
|       112|     Charger|                3000|                   0.96|
+----------+------------+--------------------+-----------------------+



rank custs within each city based on their spending

In [ ]:
joined_df = cdf.join(odf,"customer_id").join(pdf,"product_id")

grouped_df = joined_df.groupBy("city","customer_id","name").agg(sum(col("quantity")*col("price")).alias("spending"))

window_spec = Window.partitionBy("city").orderBy(col("spending").desc())

result = grouped_df.withColumn("rank",dense_rank().over(window_spec))

result.show()

+---------+-----------+-----+--------+----+
|     city|customer_id| name|spending|rank|
+---------+-----------+-----+--------+----+
|Bangalore|          3| John|    6000|   1|
|  Chennai|          5| Neha|    1000|   1|
|    Delhi|          1| Amit|   74000|   1|
|    Delhi|          6| Ravi|   25000|   2|
|Hyderabad|          4|  Ali|  140000|   1|
|Hyderabad|          8|Kiran|    2500|   2|
|   Mumbai|          2| Sara|   63000|   1|
|   Mumbai|          7|Priya|    2400|   2|
+---------+-----------+-----+--------+----+



second highest spending cust

In [ ]:
joined_df = cdf.join(odf,"customer_id").join(pdf,"product_id")

grouped_df = joined_df.groupBy("customer_id","name").agg(sum(col("quantity")*col("price")).alias("spending"))

window_spec = Window.orderBy(col("spending").desc())

result = grouped_df.withColumn("rank",dense_rank().over(window_spec))\
          .filter(col("rank")==1).drop("rank")

result.show()

+-----------+----+--------+
|customer_id|name|spending|
+-----------+----+--------+
|          4| Ali|  140000|
+-----------+----+--------+



Customer who purchased the maximum number of distinct products

In [ ]:
from pyspark.sql.functions import countDistinct

joined_df = cdf.join(odf,"customer_id").join(pdf,"product_id")

grouped_df = joined_df.groupBy("customer_id","name").agg(countDistinct("product_id").alias("products_purchased"))

window_spec = Window.orderBy(col("products_purchased").desc())

result = grouped_df.withColumn("rank",dense_rank().over(window_spec)).filter(col("rank")==1).drop("rank")

result.show()

+-----------+----+------------------+
|customer_id|name|products_purchased|
+-----------+----+------------------+
|          1|Amit|                 2|
|          2|Sara|                 2|
+-----------+----+------------------+



customers whose spending is above average customer spending

In [ ]:
joined_df = cdf.join(odf,"customer_id").join(pdf,"product_id")

grouped_df = joined_df.groupBy("customer_id","name").agg(sum(col("quantity")*col("price")).alias("spending"))

avg_spending = grouped_df.agg(avg("spending")).collect()[0][0]

result = grouped_df.filter(col("spending")>avg_spending)

result.show()





+-----------+----+--------+
|customer_id|name|spending|
+-----------+----+--------+
|          1|Amit|   74000|
|          2|Sara|   63000|
|          4| Ali|  140000|
+-----------+----+--------+



In [ ]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy()

result = grouped_df.withColumn(
    "avg_spending",
    avg("spending").over(window_spec)
).filter(
    col("spending") > col("avg_spending")
).drop("avg_spending")

result.show()

+-----------+----+--------+
|customer_id|name|spending|
+-----------+----+--------+
|          1|Amit|   74000|
|          2|Sara|   63000|
|          4| Ali|  140000|
+-----------+----+--------+



divide customers into 4 segments based on their spending from highest to lowest

In [ ]:
joined_df = cdf.join(odf,"customer_id").join(pdf,"product_id")

grouped_df = joined_df.groupBy("customer_id","name").agg(sum(col("quantity")*col("price")).alias("spending"))

window_spec = Window.orderBy("spending")

result = grouped_df.withColumn("segments",ntile(4).over(window_spec))

result.show()

+-----------+-----+--------+--------+
|customer_id| name|spending|segments|
+-----------+-----+--------+--------+
|          5| Neha|    1000|       1|
|          7|Priya|    2400|       1|
|          8|Kiran|    2500|       2|
|          3| John|    6000|       2|
|          6| Ravi|   25000|       3|
|          2| Sara|   63000|       3|
|          1| Amit|   74000|       4|
|          4|  Ali|  140000|       4|
+-----------+-----+--------+--------+

